In [1]:
import pandas as pd
import os
import run as run
import chephy_model_atchley as cpm
from tqdm import tqdm

/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def main(file_name, input_file):


    out_folder = f"../output_orr/{file_name}"
    out_name = "neighbors"
    params_file = "stats.csv"
    right = 4
    left = 4
    # label_header = args.epitope_header
    cdr3_header = "aminoAcid"


    # distances_csv = "../distance_matrix.csv"
    # distances_df = pd.read_csv(distances_csv, index_col=0)
    # substitution_matrix = {(aa1, aa2): distances_df.loc[aa1, aa2] for aa1 in distances_df.index for aa2 in distances_df.columns}
    
 
    data = run.load_dataframe(input_file)

    # data = read_csv_files_from_folder()

    data = run.prepare_data(data, cdr3_header, "label_header")    
    cdr3 = list(data[cdr3_header])
    
    sequences_set, full_to_trunc_map = cpm.truncate_sequences(cdr3, right, left)
    neighbors = cpm.find_che_phy_dist(sequences_set, full_to_trunc_map, "")

    # run.write_file(neighbors, f"{out_folder}", f"{out_name}")
    mst = run.create_mst(neighbors, f"{out_folder}/mst.csv")


    # df_results = cluster(mst, out_folder, data, label_header)
    # save_best_cluster(df_results, out_folder)
    # # think of a general method to write files

In [3]:

folder_path="/dsi/scratch/home/dsi/solefroni/orforchen/downsamples_clonotype_21355"
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
dataframes = []
num_files = len(csv_files)
i = 1
for file in tqdm(csv_files):
    # print(file, f"file {i} of {num_files}")
    full_file = os.path.join(folder_path, file)
    main(file, full_file)
    i+=1

  0%|          | 0/143 [00:00<?, ?it/s]

Truncating Sequences...


  0%|          | 0/143 [00:56<?, ?it/s]


KeyboardInterrupt: 

In [60]:
def traverse_tree(tree, sequence, max_diff, substitution_matrix, path="", current_diff=0, chephy_dist=0, depth=0, GREATEST_DISTANCE=1):
    GREATEST_DISTANCE = 72.38373253708322
    if current_diff > max_diff:
        return []

    if not tree:
        # Normalize chephy_dist by GREATEST_DISTANCE
        normalized_chephy_dist = chephy_dist / GREATEST_DISTANCE if GREATEST_DISTANCE else chephy_dist
        return [(path, current_diff, normalized_chephy_dist)] if current_diff <= max_diff else []

    matching_sequences = []
    for char, subtree in tree.items():
        if depth < len(sequence):
            seq_char = sequence[depth]
            substitution_cost = substitution_matrix.get((seq_char, char), float('inf'))  # Get cost from matrix or set to high cost if undefined
            new_diff = current_diff + (1 if char != seq_char else 0)
            new_chephy_dist = chephy_dist + (substitution_cost if char != seq_char else 0)
        else:
            new_diff = current_diff + 1  # Extra penalty for inserting a character
            new_chephy_dist = chephy_dist + substitution_matrix.get((None, char), 1)  # Cost of insertion (default 1)

        matching_sequences.extend(
            traverse_tree(subtree, sequence, max_diff, substitution_matrix, path + char, new_diff, new_chephy_dist, depth + 1, GREATEST_DISTANCE)
        )

    return matching_sequences

In [94]:
import pandas as pd

def compute_all_distances_matrix(list_of_sequences, tree, max_diff, substitution_matrix):
    GREATEST_DISTANCE = 72.38373253708322  # Normalization factor

    matrix = {}

    for seq in tqdm(list_of_sequences):
        matrix[seq] = {}
        # Get all possible distances from seq1 to seq2
        results = traverse_tree(tree, seq, max_diff, substitution_matrix)
        for result in results:
            matrix[seq][result[0]] = [result[1], result[2]]

    # Convert to Pandas DataFrame for easier visualization
    df = pd.DataFrame(matrix)
    
    return df


In [95]:
import chephy_model_tree_with_matrix as cpm
import pandas as pd
from tqdm import tqdm 


distances_csv = "../distance_matrix.csv"
distances_df = pd.read_csv(distances_csv, index_col=0)
substitution_matrix = {(aa1, aa2): distances_df.loc[aa1, aa2] for aa1 in distances_df.index for aa2 in distances_df.columns}
    

file_name = "/dsi/scratch/home/dsi/solefroni/orforchen/downsamples_clonotype_21355/STA_085.csv"
df = pd.read_csv(file_name)
sequences = df["aminoAcid"]
sequences_set, full_to_trunc_map = cpm.truncate_sequences(sequences)
sequences_list = list(sequences_set)
tree = cpm.create_tree(sequences_list)
# compute_distance_matrices(list(sequences_set), substitution_matrix, max_diff=8)
sequences_dict = {"ASNRKGPQ":[]}
compute_all_distances_matrix(sequences_list,tree, 8, substitution_matrix)

  0%|          | 76/21315 [00:16<1:17:15,  4.58it/s]


KeyboardInterrupt: 

In [1]:
import numpy as np
from scipy.spatial.distance import cdist
import pandas as pd

In [2]:
def prepare_data(data, cdr3_header):

    # understand tcrscapes data preparation


    # remove cdr3 sequences that contain non-aa letters (this also removes nan)
    amino_acid_pattern = r'^[ARNDCEQGHILKMFPSTWYV]+$'

    data_filtered = data[data[cdr3_header].str.match(amino_acid_pattern, case=False, na=False)]

    data_filtered = data_filtered[data_filtered[cdr3_header].str.len() >= 8]

    # drop duplicates
    data_filtered = data_filtered.drop_duplicates(subset=cdr3_header)

    # move truncate to here

    return data_filtered

In [3]:
sub_matrix = pd.read_csv("../distance_matrix.csv", index_col=0)
sub_matrix = sub_matrix.to_dict()

In [5]:
import chephy_model as cpm
file_name = "../files/vdjdb_score3.csv"
file_name = "/dsi/scratch/home/dsi/solefroni/orforchen/downsamples_clonotype_21355/STA_085.csv"
df = pd.read_csv(file_name)
# sequences = df["cdr3"]
# sequences = df["aminoAcid"]

cdr3_header = "aminoAcid"

# data = prepare_data(df, cdr3_header)    
# cdr3 = list(data[cdr3_header])
cdr3 = ["CASSFQGYTEAFF","CASSFQGFTEAFF"]
sequences_set, full_to_trunc_map = cpm.truncate_sequences(cdr3)
sequences = list(sequences_set)

Truncating Sequences...


In [6]:
sequences

['SSFQGFTE', 'SSFQGYTE']

In [7]:
sequences = ["AAAAAAAA","CCCCCCCC","DDDDDDDD","EEEEEEEE","FFFFFFFF","GGGGGGGG",
             "HHHHHHHH","IIIIIIII","KKKKKKKK","LLLLLLLL","MMMMMMMM","NNNNNNNN",
             "PPPPPPPP","QQQQQQQQ","RRRRRRRR","SSSSSSSS","TTTTTTTT","VVVVVVVV",
             "WWWWWWWW","YYYYYYYY"]
sequences = ['SSFQGFTE', 'SSFQGYTE']


In [8]:
import numpy as np
from scipy.spatial.distance import pdist, squareform
import pandas as pd

atchley_df = pd.read_csv("../files/atchley.csv")


# Create a dictionary mapping amino acids to their Atchley factors
atchley_dict = atchley_df.set_index('amino.acid').to_dict(orient='index')

# Convert sequences into numerical representations
sequence_vectors = []
for seq in sequences:
    seq_matrix = np.array([list(atchley_dict[aa].values()) for aa in seq])  # Extract numerical values
    seq_vector = seq_matrix.flatten()  # Flatten to a single vector
    sequence_vectors.append(seq_vector)

# Compute pairwise Euclidean distances
distance_matrix = squareform(pdist(sequence_vectors, metric='euclidean'))

# Convert to DataFrame for better readability
distance_df = pd.DataFrame(distance_matrix, index=sequences, columns=sequences)



In [9]:
distance_df

,SSFQGFTE,SSFQGYTE
SSFQGFTE,0.000000,2.545206
SSFQGYTE,2.545206,0.000000


In [20]:
import pandas as pd

# Create a DataFrame with sequences as index and columns
df = pd.DataFrame(distance_matrix, index=sequences, columns=sequences)

# Save to CSV
df.to_csv("out.csv")